In [1]:
import torch
from mmengine.config import Config, DictAction
from mmengine.runner import Runner
# from mmengine.registry import (DATA_SAMPLERS, DATASETS, EVALUATOR, FUNCTIONS,
#                                HOOKS, LOG_PROCESSORS, LOOPS, MODEL_WRAPPERS,
#                                MODELS, OPTIM_WRAPPERS, PARAM_SCHEDULERS,
#                                RUNNERS, VISUALIZERS, DefaultScope)
from mmseg.registry import MODELS


/data/JHC/openmmlab/mmengine/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


In [2]:
# 读取权重文件
weights_aug_l = torch.load('/data/JHC/openmmlab/mmsegmentation/checkpoints/InterImage/mask2former_internimage_h_896x896_80k_mapillary.pth', map_location='cpu')
# weights_atl_mmpretrain_mae_convert = torch.load('/opt/AI-Tianlong/0-ATL-paper-work/0-预训练好的权重/1-mmpretrain-vit_large_原版_epoch_50_loss0.0009-onlybackbone.pth')
# after_convert = torch.load('/opt/AI-Tianlong/0-ATL-paper-work/0-预训练好的权重/vit-adapter/mmpretrainformat-ViT-Adapter-Aug-L_16-i21k-300ep-lr_0.001-aug_medium1-wd_0.1-do_0.1-sd_0.1--imagenet2012-steps_20k-lr_0.01-res_384.pth')

/tmp/ipykernel_4160057/4134623309.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  weights_aug_l = torch.load('/data/JHC/openmmlab/mmsegmentation/checkpoints/InterImage/m

In [3]:
with open('/data/JHC/openmmlab/mmsegmentation/checkpoints/InterImage/原版.txt', 'w') as f:
    for k in weights_aug_l['state_dict'].keys():
        f.write(k + '\t' + str(weights_aug_l['state_dict'][k].shape) + '\n') 

In [4]:
# cfg_path = '/share/home/aitlong/AI-Tianlong/OpenMMLab/mmsegmentation/configs_new/ATL-paper-test-40-对比其他经典方法/20241023-60-1-segnext_mscan-l_1xb16-adamw-160k_ade20k-512x512.py'
cfg_path = '/data/JHC/openmmlab/mmsegmentation/configs_new/internImage/比赛_InterImage.py'
cfg = Config.fromfile(cfg_path)
model=cfg['model']
model_mmseg = MODELS.build(model)

/data/miniconda/miniconda3/envs/jhc-py10-2/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/data/JHC/openmmlab/mmsegmentation/mmseg/models/backbones/beit_adapter.py:278: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/data/JHC/openmmlab/mmsegmentation/mmseg/models/backbones/beit_adapter.py:295: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/data/JHC/openmmlab/mmsegmentation/ops_dcnv3/functions/dcnv3_func.py:20: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., 

using core type: DCNv3
using activation layer: GELU
using main norm layer: LN
using dpr: linear, 0.5
level2_post_norm: True
level2_post_norm_block_ids: [5, 11, 17, 23, 29]
res_post_norm: True


In [5]:
model_mmseg

EncoderDecoder(
  (data_preprocessor): SegDataPreProcessor()
  (backbone): InternImage(
    (patch_embed): StemLayer(
      (conv1): Conv2d(4, 160, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (norm1): Sequential(
        (0): to_channels_last()
        (1): LayerNorm((160,), eps=1e-06, elementwise_affine=True)
        (2): to_channels_first()
      )
      (act): GELU(approximate='none')
      (conv2): Conv2d(160, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (norm2): Sequential(
        (0): to_channels_last()
        (1): LayerNorm((320,), eps=1e-06, elementwise_affine=True)
      )
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (levels): ModuleList(
      (0): InternImageBlock(
        (blocks): ModuleList(
          (0): InternImageLayer(
            (norm1): Sequential(
              (0): LayerNorm((320,), eps=1e-06, elementwise_affine=True)
            )
            (dcn): DCNv3(
              (dw_conv): Sequential(
                (0): Conv2

In [7]:
model_mmseg.state_dict().keys()
with open('/data/JHC/openmmlab/mmsegmentation/checkpoints/InterImage/mmseg版本.txt','w') as f:
    for key in model_mmseg.state_dict().keys():
        f.write(key + '\t' + str(model_mmseg.state_dict()[key].shape) + '\n') 

In [9]:
# 读取权重文件
weights_converted = torch.load('/data/AI-Tianlong/Checkpoints/2-对比实验的权重/segnext/base/segnext_mscan_b_10channel_BGR.pth', map_location='cpu')
# weights_atl_mmpretrain_mae_convert = torch.load('/opt/AI-Tianlong/0-ATL-paper-work/0-预训练好的权重/1-mmpretrain-vit_large_原版_epoch_50_loss0.0009-onlybackbone.pth')
# after_convert = torch.load('/opt/AI-Tianlong/0-ATL-paper-work/0-预训练好的权重/vit-adapter/mmpretrainformat-ViT-Adapter-Aug-L_16-i21k-300ep-lr_0.001-aug_medium1-wd_0.1-do_0.1-sd_0.1--imagenet2012-steps_20k-lr_0.01-res_384.pth')

In [10]:
with open('./seg_next_原始权重转换10通道后.txt', 'w') as f:
    for k in weights_converted.keys():
        f.write(k + '\t' + str(weights_converted[k].shape) + '\n') 